In [1]:
# Упражнение 3: Класификация на спам съобщения
# Зареждаме dataset, обучаваме модел с scikit-learn и оценяваме резултатите

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Зареждаме SMS Spam Collection dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["label", "message"])

print(f"Размер: {len(df):,} съобщения")
df.head(10)

In [ ]:
# Разпределение на класовете
print(df["label"].value_counts())
print(f"\nДял на спам: {(df['label'] == 'spam').mean():.1%}")

In [ ]:
# Примери за ham и spam
print("=== HAM ===")
for msg in df[df["label"] == "ham"]["message"].sample(3, random_state=42):
    print(f"  {msg[:100]}")

print("\n=== SPAM ===")
for msg in df[df["label"] == "spam"]["message"].sample(3, random_state=42):
    print(f"  {msg[:100]}")

In [ ]:
# Средна дължина на съобщенията по клас
df["length"] = df["message"].str.len()

print("Средна дължина (символи):")
print(df.groupby("label")["length"].mean().round(1))

In [ ]:
# Train/test разделяне: 80% / 20%
X_train, X_test, y_train, y_test = train_test_split(
    df["message"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

print(f"Train: {len(X_train):,} съобщения")
print(f"Test:  {len(X_test):,} съобщения")

In [ ]:
# TF-IDF: превръщаме текст в числови признаци (features)
vectorizer = TfidfVectorizer(stop_words="english", max_features=100)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Признаци: {X_train_tfidf.shape[1]:,} думи")
print(f"Train матрица: {X_train_tfidf.shape}")

In [ ]:
# Обучаваме Naive Bayes класификатор
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print("Модел обучен!")
print(f"Точност (accuracy) на train: {model.score(X_train_tfidf, y_train):.3f}")
print(f"Точност (accuracy) на test:  {model.score(X_test_tfidf, y_test):.3f}")

In [ ]:
# Детайлна оценка: precision, recall, F1
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])
disp = ConfusionMatrixDisplay(cm, display_labels=["ham", "spam"])
disp.plot(cmap="Blues")
plt.title("Матрица на объркванията")
plt.show()

In [ ]:
# Тестваме с няколко нови съобщения
test_messages = [
    "Hey, are we still meeting for lunch tomorrow?",
    "CONGRATULATIONS! You won a FREE iPhone! Click here NOW!",
    "Can you pick up some milk on your way home?",
    "URGENT: Your account will be suspended. Verify your details immediately!",
]

test_tfidf = vectorizer.transform(test_messages)
predictions = model.predict(test_tfidf)

for msg, pred in zip(test_messages, predictions):
    print(f"  [{pred:4s}] {msg}")